# Notebook 02: Data Preparation
**Project 04: Predictive Maintenance for Industrial Equipment**  
**STAT 1100 -- Langara College -- Spring 2026**  
**Team:** Jongmin Lee & Aedrian

This notebook implements all 5 required data preparation tasks for the CMAPSS FD001 dataset.
Tasks are ordered by dependency: RUL labels first (needed by all subsequent steps), then
normalization, feature engineering, windowing justification, and finally train/test splitting.

**Course Coverage:** Week 3 (Data Wrangling), Week 5 (EDA), Week 6 (Normalization), Week 8 (PCA)

### Imports and Configuration

**What:** Load all required libraries for data wrangling, visualization, and machine learning preprocessing.  
**Why:** Centralizing imports at the top ensures the notebook fails fast if any dependency is missing, and makes it easy to see what tools this notebook relies on.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupShuffleSplit
import os
import json
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"pandas: {pd.__version__}, numpy: {np.__version__}, sklearn: {__import__('sklearn').__version__}")

### Load CMAPSS FD001 Data

**What:** Read the train, test, and RUL files from the CMAPSS FD001 dataset.  
**Why:** FD001 uses a single operating condition and a single fault mode (HPC degradation), making it the cleanest subset for our first predictive maintenance model. The whitespace-delimited format requires `sep=r'\s+'`.  
**Course Reference:** Week 2 -- Data Collection and Sources (importing data from text files)

In [ ]:
DATA_DIR = 'data/CMAPSSData'
columns = ['unit', 'cycle'] + [f'setting_{i}' for i in range(1, 4)] + [f'sensor_{i}' for i in range(1, 22)]

train_df = pd.read_csv(f'{DATA_DIR}/train_FD001.txt', sep=r'\s+', header=None, names=columns)
test_df = pd.read_csv(f'{DATA_DIR}/test_FD001.txt', sep=r'\s+', header=None, names=columns)
rul_df = pd.read_csv(f'{DATA_DIR}/RUL_FD001.txt', sep=r'\s+', header=None, names=['rul'])

print(f"Train: {train_df.shape}, Test: {test_df.shape}, RUL: {rul_df.shape}")
print(f"Train engines: {train_df['unit'].nunique()}, Test engines: {test_df['unit'].nunique()}")
print(f"\nTrain columns: {list(train_df.columns)}")

---
## Task 3: RUL Label Construction

**What:** Create Remaining Useful Life (RUL) labels for each row in the training data, with a piecewise-linear cap at T=130 cycles.  
**Why:** Raw linear RUL labels (e.g., 350 cycles remaining) force the model to distinguish between "very healthy" states that look identical sensor-wise. Capping at 130 focuses the model on the degradation phase where predictions are meaningful and actionable. T=130 is the CMAPSS literature standard.  
**Course Reference:** Week 3 -- Data Wrangling (data transformation, label engineering)  
**Decision Record:** See ADR-002 for detailed justification of T=130 cap.

In [ ]:
# Step 1: Compute raw RUL for training data
# For each engine, RUL = max_cycle - current_cycle
max_cycles = train_df.groupby('unit')['cycle'].max().reset_index()
max_cycles.columns = ['unit', 'max_cycle']
train_df = train_df.merge(max_cycles, on='unit')
train_df['rul'] = train_df['max_cycle'] - train_df['cycle']
train_df.drop('max_cycle', axis=1, inplace=True)

# Step 2: Apply piecewise-linear cap
RUL_CAP = 130
train_df['rul_capped'] = train_df['rul'].clip(upper=RUL_CAP)

print(f"RUL range (raw):    [{train_df['rul'].min()}, {train_df['rul'].max()}]")
print(f"RUL range (capped): [{train_df['rul_capped'].min()}, {train_df['rul_capped'].max()}]")
print(f"Rows at cap ({RUL_CAP}):  {(train_df['rul_capped'] == RUL_CAP).sum():,} / {len(train_df):,}")

### Visualize Raw vs Capped RUL

**What:** Plot the raw and capped RUL degradation curves for a single engine unit.  
**Why:** Visual confirmation that the piecewise-linear cap behaves as expected -- flat at T=130 during the healthy phase, then linearly decreasing during degradation.

In [ ]:
# Visualize raw vs capped RUL for one engine
sample = train_df[train_df['unit'] == 1]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sample['cycle'], sample['rul'], label='Raw RUL', color='lightcoral', linestyle='--')
ax.plot(sample['cycle'], sample['rul_capped'], label=f'Capped RUL (T={RUL_CAP})', color='steelblue', linewidth=2)
ax.axhline(y=RUL_CAP, color='gray', linestyle=':', alpha=0.5, label=f'Cap = {RUL_CAP}')
ax.set_xlabel('Operational Cycle')
ax.set_ylabel('Remaining Useful Life')
ax.set_title('Engine Unit 1: Raw vs Piecewise-Linear RUL')
ax.legend()
plt.tight_layout()
plt.show()

---
## Task 4: Normalisation Across Operating Conditions

**What:** Apply MinMaxScaler to sensor readings, fitting on training data only.  
**Why:** Different sensors have vastly different scales (some range 0-600, others 0-0.03). Normalization ensures all features contribute equally to distance-based models like KNN. We fit the scaler on training data only to prevent data leakage -- test data must not influence preprocessing parameters.  
**Course Reference:** Week 6 -- Classification (KNN notebook uses MinMaxScaler), Week 8 -- PCA (StandardScaler before PCA)  
**Decision Record:** See ADR-006 for normalization strategy details.

In [ ]:
# Drop constant sensors identified in EDA (sensors 1, 5, 6, 10, 16, 18, 19)
constant_sensors = ['sensor_1', 'sensor_5', 'sensor_6', 'sensor_10',
                    'sensor_16', 'sensor_18', 'sensor_19']
sensor_cols = [c for c in train_df.columns if c.startswith('sensor_')]
useful_sensors = [s for s in sensor_cols if s not in constant_sensors]
setting_cols = ['setting_1', 'setting_2', 'setting_3']

print(f"Dropped {len(constant_sensors)} constant sensors: {constant_sensors}")
print(f"Keeping {len(useful_sensors)} useful sensors: {useful_sensors}")

# MinMaxScaler -- fit on TRAINING data only
scaler = MinMaxScaler()
train_df[useful_sensors] = scaler.fit_transform(train_df[useful_sensors])
test_df[useful_sensors] = scaler.transform(test_df[useful_sensors])

print(f"\nAfter normalization (training set):")
print(f"  Min values: {train_df[useful_sensors].min().min():.4f}")
print(f"  Max values: {train_df[useful_sensors].max().max():.4f}")

---
## Task 2: Sensor Feature Engineering

**What:** Create rolling statistics (mean, standard deviation) for each sensor over a 5-cycle window.  
**Why:** Raw sensor values at a single time step don't capture the temporal degradation pattern. Rolling statistics encode how sensors are changing over recent cycles -- a rising rolling mean in temperature or a growing rolling std (more erratic readings) are strong degradation signals.  
**Course Reference:** Week 5 -- EDA (aggregation functions: mean, std, trend analysis)  
**Decision Record:** See ADR-003 for why rolling stats replace explicit windowing.

In [ ]:
ROLLING_WINDOW = 5

# For each useful sensor, compute rolling mean and rolling std
for sensor in useful_sensors:
    # Rolling mean captures the smoothed trend
    train_df[f'{sensor}_rmean'] = train_df.groupby('unit')[sensor].transform(
        lambda x: x.rolling(ROLLING_WINDOW, min_periods=1).mean()
    )
    # Rolling std captures increasing variability (degradation signal)
    train_df[f'{sensor}_rstd'] = train_df.groupby('unit')[sensor].transform(
        lambda x: x.rolling(ROLLING_WINDOW, min_periods=1).std().fillna(0)
    )

    # Same for test set
    test_df[f'{sensor}_rmean'] = test_df.groupby('unit')[sensor].transform(
        lambda x: x.rolling(ROLLING_WINDOW, min_periods=1).mean()
    )
    test_df[f'{sensor}_rstd'] = test_df.groupby('unit')[sensor].transform(
        lambda x: x.rolling(ROLLING_WINDOW, min_periods=1).std().fillna(0)
    )

# Count new features
rolling_features = [c for c in train_df.columns if '_rmean' in c or '_rstd' in c]
print(f"Created {len(rolling_features)} rolling features ({len(useful_sensors)} sensors x 2 stats)")
print(f"Total feature columns: {len(useful_sensors)} raw + {len(rolling_features)} rolling = {len(useful_sensors) + len(rolling_features)}")
print(f"\nSample rolling features for sensor_2 (engine 1, last 10 cycles):")
display(train_df[train_df['unit'] == 1][['cycle', 'sensor_2', 'sensor_2_rmean', 'sensor_2_rstd']].tail(10))

---
## Task 1: Time-Series Windowing

**What:** Justify our approach to temporal information encoding -- rolling statistics per row instead of explicit window flattening.  
**Why:** Traditional windowing (e.g., taking 30 consecutive time steps and flattening into a single feature vector) would create 30 x 14 = 420+ features from just ~17,000 usable training samples across 100 engines. This high dimensionality relative to sample count creates severe overfitting risk. Instead, our rolling statistics (mean, std over window=5) capture temporal context in just 28 additional features (14 sensors x 2 stats), keeping the feature-to-sample ratio manageable.  
**Course Reference:** Week 3 -- Data Wrangling (pipeline design), Week 7 -- Regression (feature-to-sample ratio considerations)  
**Decision Record:** See ADR-003 for the full windowing approach analysis.

In [ ]:
# Demonstrate that rolling features capture the same information as explicit windows
sample = train_df[train_df['unit'] == 1].copy()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Raw sensor vs rolling mean
axes[0].plot(sample['cycle'], sample['sensor_2'], alpha=0.5, label='Raw sensor_2', color='lightcoral')
axes[0].plot(sample['cycle'], sample['sensor_2_rmean'], label=f'Rolling mean (w={ROLLING_WINDOW})',
             color='steelblue', linewidth=2)
axes[0].set_ylabel('Normalized Value')
axes[0].set_title('Sensor 2: Raw vs Rolling Mean')
axes[0].legend()

# Rolling std shows increasing variability near failure
axes[1].plot(sample['cycle'], sample['sensor_2_rstd'], color='steelblue', linewidth=2)
axes[1].set_xlabel('Cycle')
axes[1].set_ylabel('Rolling Std')
axes[1].set_title('Sensor 2: Rolling Std (Increases Near Failure)')

plt.tight_layout()
plt.show()

print(f"Feature dimension comparison:")
print(f"  Explicit window (30 x 14 sensors): {30 * 14} features")
print(f"  Rolling stats (14 sensors x 2):     {14 * 2} features")
print(f"  Reduction: {30 * 14 - 14 * 2} fewer features -> much less overfitting risk")

---
## Task 5: Train/Validation/Test Split by Engine Unit

**What:** Split the training data into train and validation sets by engine unit (not by row), and prepare the test set.  
**Why:** Splitting by row would leak temporal information -- the model could see future sensor readings from the same engine in training and predict past readings in validation. Group-based splitting ensures complete engine trajectories stay together, preventing data leakage.  
**Course Reference:** Week 6 -- Classification (train_test_split with stratify), extending to GroupShuffleSplit for grouped data  
**Decision Record:** See ADR-006 (normalization fit on train only applies after this split).

In [ ]:
# Define feature columns for modeling
feature_cols = useful_sensors + rolling_features
print(f"Feature columns for modeling: {len(feature_cols)}")

# Split training data 80/20 by engine unit
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(splitter.split(train_df, groups=train_df['unit']))

train_set = train_df.iloc[train_idx].copy()
val_set = train_df.iloc[val_idx].copy()

# Verify no engine overlap
train_engines = set(train_set['unit'].unique())
val_engines = set(val_set['unit'].unique())
assert train_engines.isdisjoint(val_engines), "DATA LEAKAGE: engines overlap between train and val!"

print(f"Train: {len(train_set):,} rows, {len(train_engines)} engines")
print(f"Val:   {len(val_set):,} rows, {len(val_engines)} engines")
print(f"Engine overlap: {train_engines & val_engines} (should be empty)")

# Prepare test set: use last cycle of each engine
test_last = test_df.groupby('unit').last().reset_index()
# The rul_df gives RUL at the last observed cycle of each test engine
test_last['rul'] = rul_df['rul'].values
test_last['rul_capped'] = test_last['rul'].clip(upper=RUL_CAP)

print(f"\nTest:  {len(test_last)} engines (last cycle per engine)")

---
## Bonus: PCA Dimensionality Reduction

**What:** Apply PCA to reduce the sensor feature space while retaining 95% of variance.  
**Why:** With 14 raw sensors + 28 rolling features = 42 total features, some information is redundant (as shown by high correlations in EDA). PCA creates orthogonal components that maximize information retention while reducing dimensionality, potentially improving model performance and interpretability.  
**Course Reference:** Week 8 -- PCA and Factor Analysis (StandardScaler -> PCA -> variance analysis)

In [ ]:
# Standardize features before PCA (fit on training set only)
pca_scaler = StandardScaler()
X_train_scaled = pca_scaler.fit_transform(train_set[feature_cols])
X_val_scaled = pca_scaler.transform(val_set[feature_cols])
X_test_scaled = pca_scaler.transform(test_last[feature_cols])

# Fit PCA on training data, keep 95% variance
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca = pca.transform(X_val_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"PCA: {len(feature_cols)} features -> {pca.n_components_} components (95% variance retained)")
print(f"Explained variance per component: {pca.explained_variance_ratio_.round(3)}")

# Cumulative variance plot
cumvar = np.cumsum(pca.explained_variance_ratio_)
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(range(1, len(cumvar) + 1), pca.explained_variance_ratio_, alpha=0.6, label='Individual', color='steelblue')
ax.step(range(1, len(cumvar) + 1), cumvar, where='mid', label='Cumulative', color='coral', linewidth=2)
ax.axhline(y=0.95, color='gray', linestyle=':', alpha=0.5, label='95% threshold')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained Variance Ratio')
ax.set_title('PCA: Explained Variance by Component')
ax.legend()
plt.tight_layout()
plt.show()

---
## Save Processed Data

**What:** Save preprocessed datasets for the modeling notebook (Notebook 03).  
**Why:** Separating data preparation from modeling ensures reproducibility -- the modeling notebook reads clean, preprocessed data without re-running the entire pipeline. This also makes it easier for team members to work on different notebooks independently.

In [ ]:
os.makedirs('data/processed', exist_ok=True)

# Save with all features (raw + rolling + RUL)
train_set.to_csv('data/processed/train.csv', index=False)
val_set.to_csv('data/processed/val.csv', index=False)
test_last.to_csv('data/processed/test.csv', index=False)

# Save feature column list for modeling notebook
feature_info = {
    'useful_sensors': useful_sensors,
    'rolling_features': rolling_features,
    'feature_cols': feature_cols,
    'constant_sensors': constant_sensors,
    'rul_cap': RUL_CAP,
    'rolling_window': ROLLING_WINDOW
}
with open('data/processed/feature_info.json', 'w') as f:
    json.dump(feature_info, f, indent=2)

print("=== Saved Files ===")
for fname in ['train.csv', 'val.csv', 'test.csv', 'feature_info.json']:
    fpath = f'data/processed/{fname}'
    size = os.path.getsize(fpath)
    print(f"  {fname}: {size:,} bytes")

print(f"\n=== Data Shapes ===")
print(f"  Train: {train_set.shape}")
print(f"  Val:   {val_set.shape}")
print(f"  Test:  {test_last.shape}")
print(f"  Features: {len(feature_cols)}")

---
## Data Preparation Pipeline Summary

| Task | What We Did | Key Parameters |
|------|-------------|----------------|
| Task 3: RUL Labels | Piecewise-linear with cap | T = 130 cycles |
| Task 4: Normalisation | MinMaxScaler (fit on train only) | Dropped 7 constant sensors |
| Task 2: Feature Engineering | Rolling mean + std | Window = 5 cycles |
| Task 1: Windowing | Rolling stats replace explicit windows | 42 features vs 420+ |
| Task 5: Train/Val Split | GroupShuffleSplit by engine unit | 80/20 split, no leakage |
| Bonus: PCA | Dimensionality reduction | 95% variance retained |

**Output files:** `data/processed/{train.csv, val.csv, test.csv, feature_info.json}`

**Next:** Notebook 03 -- Model Building (regression, classification, sensor importance, alarm threshold)